In [11]:
import json

# load final features — defined in feature engineering notebook
# ensures modelling uses exact same features as feature engineering
with open('../models/final_features.json') as f:
    final_features = json.load(f)

X = fraud_df[final_features]
y = fraud_df['Class']

print(f"Features loaded: {len(final_features)}")
print(f"Features: {final_features}")

Features loaded: 30
Features: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V27', 'V28', 'Amount_log', 'Hour', 'rolling_std_amount']



Train:    (199364, 30)
Validate: (42721, 30)
Test:     (42722, 30)


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

# scale features — required for logistic regression
scaler = StandardScaler()
X_train_scaled    = scaler.fit_transform(X_train)
X_validate_scaled = scaler.transform(X_validate)
X_test_scaled     = scaler.transform(X_test)

# class weight for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

# 1. Logistic Regression
print("\nTraining Logistic Regression...")
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    C=0.1,
    solver='lbfgs',
    random_state=42
)
lr.fit(X_train_scaled, y_train)
print("Logistic Regression Done ")

# 2. Random Forest
print("\nTraining Random Forest...")
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print("Random Forest Done ")

# 3. XGBoost
print("\nTraining XGBoost...")
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    random_state=42,
    eval_metric='aucpr',
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
print("XGBoost Done ")

scale_pos_weight: 518.2

Training Logistic Regression...
Logistic Regression Done 

Training Random Forest...
Random Forest Done 

Training XGBoost...
XGBoost Done 


In [14]:
import joblib
import os
from datetime import datetime

os.makedirs('../models', exist_ok=True)

# save models — overwrites previous version
joblib.dump(lr, '../models/logistic_regression.pkl')
joblib.dump(rf, '../models/random_forest.pkl')
joblib.dump(xgb_model, '../models/xgboost.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

# save hyperparameters with timestamp
import json
hyperparameters = {
    'trained_at': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'n_features': X_train.shape[1],
    'train_size': X_train.shape[0],
    'logistic_regression': {
        'class_weight': 'balanced',
        'C': 0.1,
        'max_iter': 1000,
        'solver': 'lbfgs'
    },
    'random_forest': {
        'n_estimators': 200,
        'max_depth': 10,
        'min_samples_split': 10,
        'min_samples_leaf': 5,
        'max_features': 'sqrt',
        'class_weight': 'balanced'
    },
    'xgboost': {
        'scale_pos_weight': round(scale_pos_weight, 2),
        'n_estimators': 200,
        'max_depth': 6,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 5,
        'gamma': 1,
        'eval_metric': 'aucpr'
    }
}

with open('../models/hyperparameters.json', 'w') as f:
    json.dump(hyperparameters, f, indent=4)

print("Models saved ")
print("Hyperparameters saved ")
print(f"Trained at: {hyperparameters['trained_at']}")
print(f"Features: {hyperparameters['n_features']}")

Models saved 
Hyperparameters saved 
Trained at: 2026-05-14 21:35
Features: 30


In [15]:
# save validate and test sets for evaluation notebook
joblib.dump(X_validate, '../models/X_validate.pkl')
joblib.dump(X_test, '../models/X_test.pkl')
joblib.dump(y_validate, '../models/y_validate.pkl')
joblib.dump(y_test, '../models/y_test.pkl')
joblib.dump(X_validate_scaled, '../models/X_validate_scaled.pkl')
joblib.dump(X_test_scaled, '../models/X_test_scaled.pkl')

print("Validate and test sets saved ")

Validate and test sets saved 
